**Before starting this practical session**:

1. **Change the runtime type on Google Colab**: `Runtime > Change runtime type > T4 GPU`
2. Run the cells **in order**. Each part frees the GPU memory used by the previous part: the free version of Colab has about 15 GB of GPU memory (VRAM) and 12 GB of RAM, which is not enough to keep all the models loaded at the same time.

*Remark*: a warning about `HF_TOKEN` may appear when the models are downloaded. It can be ignored: all the models used here are public and can be downloaded without a Hugging Face account.

# Inference with pretrained models

### Introduction

Many pretrained models are publicly available on the internet.
Some are quite general, but most of them are models fine-tuned on specific tasks.

[Hugging Face](https://huggingface.co) is a platform that hosts models, datasets and demos.
Several Python packages, such as [transformers](https://huggingface.co/docs/transformers/index) and [diffusers](https://huggingface.co/docs/diffusers/index), have been developed in this ecosystem and provide pretrained models through a common application programming interface (API).

The list of all the models available on Hugging Face is available [here](https://huggingface.co/models).
To date, there are more than one million pretrained models available.
It is easy to filter the list of models according to some criteria (task, library, dataset, language, license, etc.).

In this notebook, we will not train any model.
We will only perform inference with pretrained, publicly available models.
The goal of this notebook is to learn how to search for pretrained models and how to use them without a graphical interface (such as ChatGPT).
Another advantage of pretrained, publicly available models is that the only limit to their use is computing power.

## Installation and configuration

`transformers` and `torch` are already installed on Colab. We update `diffusers` so that it is compatible with the installed version of `transformers`.

We also uninstall `torchao` (a quantization library preinstalled on Colab, not needed here): its version is too old for the latest version of `diffusers`, which tries to import it automatically and crashes with the error `cannot import name 'FqnToConfig' from 'torchao.quantization'`.

> If you had already imported `diffusers` before running this cell, restart the session (`Runtime > Restart session`) and continue from the next cell.

In [ ]:
%pip uninstall -y -q torchao
%pip install -q -U diffusers accelerate

In [ ]:
import gc

import torch
import transformers
import diffusers

print(f"transformers: {transformers.__version__}")
print(f"diffusers:    {diffusers.__version__}")
print(f"torch:        {torch.__version__}")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (check the runtime type)"
print(f"GPU:          {gpu}")

device = "cuda" if torch.cuda.is_available() else "cpu"


def free_memory(*names):
    '''Delete the given variables (models) and free the GPU memory.'''
    for name in names:
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Question answering

A [question answering system](https://en.wikipedia.org/wiki/Question_answering) is a computer system that automatically answers questions asked by humans in natural language.

We will use the [RoBERTa-base-squad2](https://huggingface.co/deepset/roberta-base-squad2) model, which is the [RoBERTa](https://huggingface.co/FacebookAI/roberta-base) model fine-tuned on the [SQuAD2.0](https://huggingface.co/datasets/rajpurkar/squad_v2) dataset.
This model was trained on pairs of questions and answers, including questions that cannot be answered.

It is an **extractive** model: for each token of the context, the model predicts the probability that this token is the start of the answer and the probability that it is the end.
The returned answer is the span of the context that maximizes the product of these two probabilities.

> **Remark**: up to version 4 of `transformers`, one could directly use `pipeline('question-answering', ...)`.
> This pipeline was removed in version 5. We therefore load the tokenizer and the model with the `AutoTokenizer` and `AutoModelForQuestionAnswering` classes, then write a small `question_answering` function that reproduces the behavior of the former pipeline.

The input of the function is a dictionary with the following key-value pairs:
* the key `'question'`, whose value is a string corresponding to the question,
* the key `'context'`, whose value is a string containing the information needed to answer the question.

The output of the function is a dictionary with the following key-value pairs:
* the key `'score'`, whose value is the confidence score of the model in its answer,
* the key `'start'`, whose value is the index of the first character of the answer,
* the key `'end'`, whose value is the index following the last character of the answer (as for slicing in Python: `context[start:end]`),
* the key `'answer'`, whose value is the answer returned by the model.

Here is an example:

```py
>>> question_answering({
...     'question': "How many planets are there in the solar system?",
...     'context': (
...         "The eight planets of the solar system are Mercury, Venus, Earth, "
...         "Mars, Jupiter, Saturn, Uranus, and Neptune."
...     )
... })
{'score': 0.79..., 'start': 4, 'end': 9, 'answer': 'eight'}
```

An important limitation of the model is that the answer must appear explicitly (i.e., as words) in the context.
For instance, if we remove the word `'eight'` from the context, the answer is disappointing (but the confidence score is also much lower).

```py
>>> question_answering({
...     'question': "How many planets are there in the solar system?",
...     'context': (
...         "The planets of the solar system are Mercury, Venus, Earth, "
...         "Mars, Jupiter, Saturn, Uranus, and Neptune."
...     )
... })
{'score': 0.01...,
 'start': 46,
 'end': 102,
 'answer': 'Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune'}
```

### Exercise

Use this model with your own pairs of questions and contexts.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_name_qa = "deepset/roberta-base-squad2"
tokenizer_qa = AutoTokenizer.from_pretrained(model_name_qa)
model_qa = AutoModelForQuestionAnswering.from_pretrained(model_name_qa).to(device).eval()

In [ ]:
def question_answering(inputs, max_answer_length=30):
    """Reproduce the question-answering pipeline: return the span of the context
    with the highest probability of being the answer to the question."""
    question, context = inputs["question"], inputs["context"]
    encoding = tokenizer_qa(
        question, context, truncation="only_second", max_length=384,
        return_offsets_mapping=True, return_tensors="pt",
    )
    offsets = encoding.pop("offset_mapping")[0]
    sequence_ids = encoding.sequence_ids(0)
    encoding = encoding.to(model_qa.device)
    with torch.no_grad():
        outputs = model_qa(**encoding)
    # Keep only the tokens of the context (the answer must be in it)
    is_context = torch.tensor([s == 1 for s in sequence_ids])
    start_logits = outputs.start_logits[0].cpu().masked_fill(~is_context, float("-inf"))
    end_logits = outputs.end_logits[0].cpu().masked_fill(~is_context, float("-inf"))
    start_probs, end_probs = start_logits.softmax(-1), end_logits.softmax(-1)
    # Score of each span (start, end) with start <= end and a limited length
    scores = torch.triu(start_probs[:, None] * end_probs[None, :])
    scores = torch.tril(scores, diagonal=max_answer_length - 1)
    best = scores.argmax()
    start_token, end_token = divmod(best.item(), scores.shape[1])
    start, end = offsets[start_token][0].item(), offsets[end_token][1].item()
    return {"score": scores[start_token, end_token].item(), "start": start, "end": end, "answer": context[start:end]}

In [ ]:
question_answering({
    'question': "How many planets are there in the solar system?",
    'context': (
        "The eight planets of the solar system are Mercury, Venus, Earth, "
        "Mars, Jupiter, Saturn, Uranus, and Neptune."
    )
})

In [ ]:
question_answering({
    'question': "How many planets are there in the solar system?",
    'context': (
        "The planets of the solar system are Mercury, Venus, Earth, "
        "Mars, Jupiter, Saturn, Uranus, and Neptune."
    )
})

In [ ]:
question_answering({
    'question': "Where am I teaching?",
    'context': "My name is Johann and I'm an assistant professor of computer science at ENSAI."
})

In [ ]:
question_answering({
    'question': "What is Geralt killing?",
    'context': (
        "Geralt is a Witcher, a man whose magic powers, enhanced by long "
        "training and a mysterious elixir, have made him a brilliant fighter "
        "and a merciless hunter. Yet he is no ordinary killer: his targets are "
        "the multifarious monsters and vile fiends that ravage the land "
        "and attack the innocent."
    )
})

In [ ]:
question_answering({
    'question': "Where is Aloy going?",
    'context': (
        "Horizon Forbidden West continues the story of Aloy, a young hunter of the "
        "Nora tribe and a clone of the Old World scientist Elisabet Sobeck, as she "
        "leads a band of companions on a quest to the frontier region known as the "
        "Forbidden West to find the source of a mysterious plague that kills all it "
        "infects. On her journey across these lands, Aloy encounters new regions "
        "ravaged by massive storms and deadly machines, and conflicts with a tribe "
        "of nomadic raiders, some of which have tamed the machines as war mounts. "
        "She discovers a vast array of environments and ecosystems, including lush "
        "valleys, dry deserts, snowy mountains, tropical beaches, ruined cities, "
        "and underwater settings."
    )
})

In [ ]:
# Free the memory before moving on to the next part
free_memory("model_qa", "tokenizer_qa")

## Text-to-image generation

A [text-to-image model](https://en.wikipedia.org/wiki/Text-to-image_model) is a model that creates images, including artworks, from a description in natural language (a *prompt*).
Such models started to be developed in the mid-2010s.
Today, the images returned by models such as [DALL-E](https://openai.com/index/dall-e-3/), [Imagen](https://deepmind.google/models/imagen/), [Stable Diffusion](https://stability.ai/stable-image) and [Midjourney](https://www.midjourney.com/explore) are close to real photographs or to images drawn by a human artist.

The three models used below are fine-tuned versions of Stable Diffusion (about 1 billion parameters). In half precision (`torch.float16`), each of them takes about 2.5 to 3 GB of GPU memory, which easily fits on a T4 GPU.

*Remarks*:
* The T4 GPU does not handle the `bfloat16` format efficiently: we therefore use `torch.float16`.
* To save Colab's GPU quota, we use the *DPM++ 2M Karras* sampler with 25 denoising steps instead of the default 50 steps, for a similar quality.
* The `generator` argument sets the random seed to obtain reproducible images. Change the seed (or remove this argument) to obtain other images.
* One model per part is loaded, then deleted before the next part.

### Mosaic images

We will use the [MosaicArt](https://huggingface.co/Guizmus/MosaicArt) model to generate mosaic images.
This model is based on the Stable Diffusion 2.1 model (resolution 768 × 768), fine-tuned with [DreamBooth](https://dreambooth.github.io).

Below are some images generated by this model.

[<img src="https://huggingface.co/Guizmus/MosaicArt/resolve/main/showcase.jpg" width="450"/>](https://huggingface.co/Guizmus/MosaicArt/resolve/main/showcase.jpg)

Here is an example of how to use this model. The token `"Mosaic Art"` at the beginning of the prompt indicates that we want to generate mosaic images.

```py
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch

# Define the generator
mosaic_art_generator = StableDiffusionPipeline.from_pretrained(
    "Guizmus/MosaicArt", torch_dtype=torch.float16
).to("cuda")
mosaic_art_generator.scheduler = DPMSolverMultistepScheduler.from_config(
    mosaic_art_generator.scheduler.config, use_karras_sigmas=True
)

# Generate an image given a prompt and show it
prompt = "Mosaic Art dog on the moon"
image = mosaic_art_generator(prompt, num_inference_steps=25).images[0]
image
```

### Exercise

Use this pretrained model to generate mosaic images.

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

mosaic_art_generator = StableDiffusionPipeline.from_pretrained(
    "Guizmus/MosaicArt", torch_dtype=torch.float16
).to(device)
mosaic_art_generator.scheduler = DPMSolverMultistepScheduler.from_config(
    mosaic_art_generator.scheduler.config, use_karras_sigmas=True
)

In [ ]:
prompt = "Mosaic Art dog on the moon"
image = mosaic_art_generator(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
prompt = "Mosaic Art cat in a forest"
image = mosaic_art_generator(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
prompt = "Mosaic Art lions in the wild"
image = mosaic_art_generator(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
free_memory("mosaic_art_generator")

### Images in the style of Hergé

[Hergé](https://en.wikipedia.org/wiki/Herg%C3%A9) is a Belgian comic book author, best known for *The Adventures of Tintin*.
You will generate images in the style of Hergé.

Once again, the principle is the same.
This is a Stable Diffusion model (version 1.x, resolution 512 × 512) fine-tuned with [DreamBooth](https://dreambooth.github.io) to generate images in the style of Hergé.
You can find the 10 original images used for fine-tuning [here](https://huggingface.co/sd-dreambooth-library/herge-style).

Here is an example of how to use this model. The keyword in the prompt is `sks herge_style` (the model was fine-tuned with the prompt `"a photo of sks herge_style"`).

```py
from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler
import torch

# Load the model
pipeline = AutoPipelineForText2Image.from_pretrained(
    "sd-dreambooth-library/herge-style", torch_dtype=torch.float16
).to("cuda")
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
    pipeline.scheduler.config, use_karras_sigmas=True
)

# Define the prompt and generate an image
prompt = (
    "A cute sks herge_style brown bear eating a slice of pizza, "
    "stunning color scheme, masterpiece, illustration"
)
image = pipeline(prompt, num_inference_steps=25).images[0]
image
```

*Remark*: this model includes a safety checker that sometimes wrongly returns an entirely black image. In that case, run the generation again with another seed.

### Exercise

Use this pretrained model to generate images in the style of Hergé.

In [ ]:
from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler

pipeline = AutoPipelineForText2Image.from_pretrained(
    "sd-dreambooth-library/herge-style", torch_dtype=torch.float16
).to(device)
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
    pipeline.scheduler.config, use_karras_sigmas=True
)

In [ ]:
prompt = (
    "A cute sks herge_style brown bear eating a slice of pizza, "
    "stunning color scheme, masterpiece, illustration"
)
image = pipeline(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
prompt = (
    "A white sks herge_style dog biting the leg of a man, "
    "stunning color scheme, masterpiece, illustration"
)
image = pipeline(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
prompt = (
    "A black sks herge_style cat walking in a garden, "
    "stunning color scheme, masterpiece, illustration"
)
image = pipeline(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
free_memory("pipeline")

### Images generated by Openjourney

[Openjourney](https://huggingface.co/prompthero/openjourney) is an open-source model based on the [Stable Diffusion](https://en.wikipedia.org/wiki/Stable_Diffusion) model (version 1.5) and fine-tuned on images generated by [Midjourney](https://en.wikipedia.org/wiki/Midjourney), which is also an image generator. The token `mdjrny-v4 style` must be added to the prompt.

Here is an example of how to use this model.

```py
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch

# Load the model
model_id = "prompthero/openjourney"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True
)

# Generate an image
prompt = "retro yellow car in a car park, mdjrny-v4 style"
image = pipe(prompt, num_inference_steps=25).images[0]
image
```

### Exercise

Use this pretrained model to generate graphic art images.

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

model_id = "prompthero/openjourney"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True
)

In [ ]:
prompt = "retro yellow car in a car park, mdjrny-v4 style"
image = pipe(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
prompt = "man sitting on a bench in a park, mdjrny-v4 style"
image = pipe(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
prompt = "lions in the wild, mdjrny-v4 style"
image = pipe(
    prompt, num_inference_steps=25, generator=torch.Generator(device).manual_seed(0)
).images[0]
image

In [ ]:
free_memory("pipe")

## Translation

[Translation](https://en.wikipedia.org/wiki/Translation) is the process of conveying the meaning of a text written in one language in another language.
[Machine translation](https://en.wikipedia.org/wiki/Machine_translation) refers to the raw translation of a text entirely performed by one or more computer programs.
[Neural machine translation](https://en.wikipedia.org/wiki/Neural_machine_translation) is an approach to machine translation that uses an artificial neural network to predict the likelihood of a sequence of words, typically modeling and translating entire sentences with a single model.

Machine translation tools such as [DeepL Translator](https://www.deepl.com/translator) and [Google Translate](https://translate.google.com) are based on artificial neural networks.

We will use a smaller version of the pretrained [Text-To-Text Transfer Transformer](https://huggingface.co/google-t5/t5-small) model (T5-small, 60 million parameters).
Although this model can be used for [many natural language processing tasks](https://huggingface.co/docs/transformers/model_doc/t5#resources), we will focus on translation only.
This smaller model works for four languages: English, French, Romanian and German.

Here is an example of how to use this model:

```py
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the model
tokenizer_t5 = AutoTokenizer.from_pretrained("google-t5/t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-small")

# Define the input
sentence = "Iris is a flowering plant genus of 310 accepted species with showy flowers."

# Define the task
input_ids = tokenizer_t5(f"translate English to French: {sentence}", return_tensors="pt").input_ids

# Generate the output
outputs = model_t5.generate(input_ids, max_new_tokens=50)

# Print the output
print(tokenizer_t5.decode(outputs[0], skip_special_tokens=True))
```

Pay particular attention to three elements:

* The beginning of the prompt indicates the task: `'translate English to French: '` to translate from English to French.
* English is the only available language for the sentence to translate. There are therefore three possible tasks: English > French, English > German and English > Romanian.
* `max_new_tokens` indicates the maximum number of tokens to generate.

### Exercise

Use this pretrained model to translate sentences from one language to another among the languages available with this model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_t5 = AutoTokenizer.from_pretrained("google-t5/t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-small").to(device)


def translate(sentence, target_language="French", max_new_tokens=50):
    '''Translate an English sentence into French, German or Romanian.'''
    input_ids = tokenizer_t5(
        f"translate English to {target_language}: {sentence}", return_tensors="pt"
    ).input_ids.to(device)
    outputs = model_t5.generate(input_ids, max_new_tokens=max_new_tokens)
    return tokenizer_t5.decode(outputs[0], skip_special_tokens=True)

In [ ]:
sentence = "Iris is a flowering plant genus of 310 accepted species with showy flowers."

input_ids = tokenizer_t5(f"translate English to French: {sentence}", return_tensors="pt").input_ids.to(device)
outputs = model_t5.generate(input_ids, max_new_tokens=50)
print(tokenizer_t5.decode(outputs[0], skip_special_tokens=True))

In [ ]:
sentence = "Time flies by so fast."
print(translate(sentence, target_language="French"))

In [ ]:
sentence = "Time flies by so fast."
print(translate(sentence, target_language="German"))

In [ ]:
sentence = "Time flies by so fast."
print(translate(sentence, target_language="Romanian"))

In [ ]:
free_memory("model_t5", "tokenizer_t5")

## Music generation

Artificial intelligence is increasingly used in the music industry.

Here, we will use the [MusicGen](https://huggingface.co/facebook/musicgen-small) model, and more precisely its smallest version (300 million parameters).

The model generates 50 audio tokens per second: the `max_new_tokens` argument therefore controls the duration of the track (e.g., 256 tokens ≈ 5 seconds). The maximum duration is about 30 seconds (1503 tokens), but the longer the track, the longer the generation takes.

Here is an example of how to use this model:
```python
from transformers import pipeline
import IPython.display
import scipy

# Load the model
synthesiser = pipeline("text-to-audio", "facebook/musicgen-small", device="cuda")

# Define the prompt
prompt = "lo-fi music with a soothing melody"

# Generate the sample
music = synthesiser(prompt, generate_kwargs={"do_sample": True, "max_new_tokens": 256})

# Save the result as a WAV file
scipy.io.wavfile.write("lo-fi.wav", rate=music["sampling_rate"], data=music["audio"])

# Play the result
IPython.display.Audio(music["audio"], rate=music["sampling_rate"])
```

### Exercise

Use this pretrained model to generate different pieces of music.

In [ ]:
from transformers import pipeline
import IPython.display
import scipy

# Load the model
synthesiser = pipeline("text-to-audio", "facebook/musicgen-small", device=device)


def generate_music(prompt, max_new_tokens=256, filename=None):
    '''Generate a track from a prompt, save it as a WAV file and display it.'''
    music = synthesiser(prompt, generate_kwargs={"do_sample": True, "max_new_tokens": max_new_tokens})
    if filename is None:
        filename = "".join(c if c.isalnum() else "_" for c in prompt)[:50] + ".wav"
    scipy.io.wavfile.write(filename, rate=music["sampling_rate"], data=music["audio"])
    print(f"File saved: {filename}")
    return IPython.display.Audio(music["audio"], rate=music["sampling_rate"])

In [ ]:
generate_music("lo-fi music with a soothing melody")

In [ ]:
generate_music("80s pop track with bassy drums and synth")